In [2]:
# 1. Install Dependencies
!pip install torch_geometric networkx
!pip install optional_dependencies torch_scatter torch_sparse torch_cluster torch_spline_conv -f https://data.pyg.org/whl/torch-2.3.0+cu121.html

import os
import torch
import numpy as np
import pandas as pd
import networkx as nx
from google.colab import drive
from torch_geometric.data import Data
from sklearn.model_selection import train_test_split

# 2. Mount Drive
drive.mount('/content/drive')

# 3. Directory Configurations
base_dir = '/content/drive/MyDrive/ASD_GNN_Research2'
raw_data_path = os.path.join(base_dir, 'raw_data')
conn_dir = os.path.join(base_dir, 'connectivity')
ts_dir = os.path.join(base_dir, 'preprocessed', 'time_series')
graphs_base_dir = os.path.join(base_dir, 'graphs')

pheno_clean = pd.read_csv(os.path.join(raw_data_path, 'phenotypic_cleaned.csv'))

# 4. Parameters
CORRELATION_THRESHOLD = 0.3
graph_list = []

print("Engineering advanced graphs with Topological & Biological features...")

for idx, row in pheno_clean.iterrows():
    subject_id = row['SUB_ID']
    label = row['DX_GROUP']

    matrix_path = os.path.join(conn_dir, f'sub_{subject_id}_conn.npy')
    ts_path = os.path.join(ts_dir, f'sub_{subject_id}_ts.npy')

    if not os.path.exists(matrix_path) or not os.path.exists(ts_path):
        continue

    # --- EDGE CONSTRUCTION ---
    matrix = np.load(matrix_path)

    # Create adjacency mask
    adj_mask = (matrix > CORRELATION_THRESHOLD) & (~np.eye(matrix.shape[0], dtype=bool))
    edge_indices = np.where(adj_mask)

    edge_index = torch.tensor(np.array(edge_indices), dtype=torch.long)
    edge_attr = torch.tensor(matrix[edge_indices], dtype=torch.float).unsqueeze(1)

    # --- TOPOLOGICAL NODE FEATURES (NetworkX) ---
    # Build a NetworkX graph from the adjacency mask to calculate graph theory metrics
    nx_graph = nx.from_numpy_array(adj_mask.astype(int))

    # Calculate metrics (returns dictionaries)
    deg_cent = nx.degree_centrality(nx_graph)
    bet_cent = nx.betweenness_centrality(nx_graph)
    clust_coeff = nx.clustering(nx_graph)

    # Convert dictionaries to arrays ordered by node index (0 to 115)
    node_deg = np.array([deg_cent[i] for i in range(116)])
    node_bet = np.array([bet_cent[i] for i in range(116)])
    node_clust = np.array([clust_coeff[i] for i in range(116)])

    # --- BIOLOGICAL NODE FEATURES ---
    ts_data = np.load(ts_path)
    node_var = np.var(ts_data, axis=0)
    node_std = np.std(ts_data, axis=0)
    node_max = np.max(np.abs(ts_data), axis=0)

    # --- COMBINE ALL 6 FEATURES ---
    # Stack into a [116, 6] feature matrix
    x = np.vstack([node_var, node_std, node_max, node_deg, node_bet, node_clust]).T
    x = torch.tensor(x, dtype=torch.float)

    # --- LABEL & DATA OBJECT ---
    y = torch.tensor([label], dtype=torch.long)
    graph_obj = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)
    graph_list.append((subject_id, graph_obj))

print(f"Generated {len(graph_list)} advanced graphs with 6 feature dimensions.")

# 5. Stratified Splitting and Serialization
labels = [g[1].y.item() for g in graph_list]
train_val, test_data = train_test_split(graph_list, test_size=0.15, stratify=labels, random_state=42)
train_labels = [g[1].y.item() for g in train_val]
train_data, val_data = train_test_split(train_val, test_size=0.1765, stratify=train_labels, random_state=42)

splits = {'train': train_data, 'val': val_data, 'test': test_data}

for split_name, dataset in splits.items():
    split_path = os.path.join(graphs_base_dir, split_name)
    print(f"Overwriting {len(dataset)} graphs in graphs/{split_name}...")
    for subject_id, data_obj in dataset:
        torch.save(data_obj, os.path.join(split_path, f'sub_{subject_id}.pt'))

print("\nGraph engineering pipeline complete! Your `.pt` files now contain advanced graph theory metrics.")

Looking in links: https://data.pyg.org/whl/torch-2.3.0+cu121.html
Mounted at /content/drive
Engineering advanced graphs with Topological & Biological features...
Generated 871 advanced graphs with 6 feature dimensions.
Overwriting 609 graphs in graphs/train...
Overwriting 131 graphs in graphs/val...
Overwriting 131 graphs in graphs/test...

Graph engineering pipeline complete! Your `.pt` files now contain advanced graph theory metrics.
